In [ ]:
pip install numpy scipy sounddevice librosa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob

import librosa
import librosa.display
import IPython.display as ipd

In [ ]:
audio_files = glob(r"dataset path")
print(audio_files)
print(len(audio_files))

In [ ]:
def Trim_Pad_Standarize(audio_files, frame_length=2048, hop_length=512, fixed_sample_rate = 48000, fixed_size=1.5):

    AF = audio_files
    raw_datas = []
    trim_datas = []
    start_samples = []
    end_samples = []

    fixed_length = int(fixed_size*fixed_sample_rate)

    for i in range(len(AF)):

        raw_data, sr = librosa.load(
            audio_files[i], 
            sr = fixed_sample_rate,
            mono = True
        )

        raw_datas.append(raw_data)

        # sample_rates.append(sample_rate)

        rms = librosa.feature.rms(
            y = raw_data,
            frame_length=frame_length,
            hop_length=hop_length
        )[0]

        times = librosa.frames_to_samples(
            np.arange(len(rms)),
            hop_length=hop_length
        )

        noise_floor = np.percentile(rms, 15)
        # threshold = noise_floor*2
        threshold = max(noise_floor*2, np.mean(rms))

        spike = np.where(rms > threshold)[0]

        if len(spike) == 0:
            trim_datas.append(np.zeros(fixed_length))
            start_samples.append(0)
            end_samples.append(fixed_length)
            continue

        start_frame = spike[0]
        end_frame = spike[-1]

        start_sample = times[start_frame]
        end_sample = min(times[end_frame] + frame_length, len(raw_data))

        onsets = librosa.onset.onset_detect(
            y = raw_data,
            sr = fixed_sample_rate,
            units = 'samples'
        )

        if len(onsets) > 0:
            nearest_onset = onsets[np.argmin(np.abs(onsets - start_sample))]
            start_sample = min(start_sample, nearest_onset)

        margin = int(0.01*sr)
        start_sample = max(0, start_sample - margin)
        end_sample = min(len(raw_data), end_sample + margin)

        start_samples.append(start_sample)
        end_samples.append(end_sample)

        RawData_trimmed = raw_data[start_sample:end_sample]

        if len(RawData_trimmed) < fixed_length:
            RawData_trimmed = np.pad(RawData_trimmed, (0, fixed_length - len(RawData_trimmed)))
        else:
            RawData_trimmed = RawData_trimmed[:fixed_length]
        
        trim_datas.append(RawData_trimmed)
    
    return raw_datas, trim_datas, start_samples, end_samples

In [ ]:
audio_files = glob(r"C:\Users\jashw\OneDrive\Desktop\Sem 4\Robotics\Project\Dataset\Audio data\AISI 304 steel\*")
# print(audio_files)

R, T, SS, ES = Trim_Pad_Standarize(audio_files)
print("length Raw data : ", len(R))
print("length Trimmed data : ", len(T))

fig, axes = plt.subplots(2, 1, figsize=(10,7))

# plt.figure(figsize=(10,4))

axes[0].plot(R[0], alpha=0.4, label="Original")
axes[0].axvline(SS[0], color='g', linestyle='--',label='Start')
axes[0].axvline(ES[0], color='r', linestyle='--',label='End')
axes[0].legend()
axes[0].set_title("Detected ringing region")

# plt.figure(figsize=(10,4))

axes[1].plot(T[0])
axes[1].set_title("Trimmed metal ringing")

plt.tight_layout()
plt.show()

In [ ]:
def rms_normalization(trims_datasets, target_rms = 0.1):

    TD = trims_datasets

    for i in range(len(TD)):
        TD[i] = TD[i]*(target_rms/(np.sqrt(np.mean(TD[i]**2)) + 1e-8))

    return TD

def peak_normalization(trims_datasets):

    TD = trims_datasets

    for i in range(len(TD)):
        TD[i] = TD[i]/np.max(np.abs(TD[i]) + 1e-8)
    
    return TD


With RMS Normalization

In [ ]:
T_Rnorm = rms_normalization(T)
# print(T_Rnorm.shape)

plt.plot(T_Rnorm[0])
plt.title("Normalized")
plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

def bandpass(Trimmed_norm_data, fixed_sample_rate, low = 200, high = 8000):

    b, a = butter(
        4,
        [low*2/(fixed_sample_rate), high*2/(fixed_sample_rate)],
        btype = 'band'
    )

    return filtfilt(b, a, Trimmed_norm_data)

In [ ]:
TR_denoising = bandpass(T_Rnorm, 48000)
print(TR_denoising.shape)

plt.plot(TR_denoising[0])
plt.title("Denoised")
plt.show()

In [ ]:
def pre_emphasis(Denoised_data, coef = 0.97):

    DD = Denoised_data
    for i in range(len(DD)):
        DD[i] = librosa.effects.preemphasis(DD[i], coef=coef)
    
    return DD

In [ ]:
PE_RMS_data = pre_emphasis(TR_denoising)

plt.plot(PE_RMS_data[0])
plt.title("Denoised")
plt.show()

In [ ]:
def Feature_Extraction(Final_trim_data, sample_rate=48000, frame_length=2048, hop_length=512, n_mels=128):

    FD =  librosa.feature.melspectrogram(
        y = Final_trim_data,
        sr = sample_rate,
        n_fft = frame_length,
        hop_length = hop_length,
        n_mels = n_mels
    )

    FD_PtoD = librosa.power_to_db(FD)

    return FD_PtoD


In [ ]:
FE_R = Feature_Extraction(PE_RMS_data)

plt.plot(FE_R[0])
plt.title("Feature Extraction")
plt.show()